### **IMPORTO LIBRERIE**

In [ ]:
from pathlib import Path

import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

### **IMPORTO DATI**

In [ ]:
processed_dir = Path("datas/processed")

software_df = pd.read_csv(processed_dir / "dataset_software.csv")
hardware_df = pd.read_csv(processed_dir / "dataset_hardware.csv")

### **REGRESSIONE OLS POOLED ROE**

Stima pooled su imprese software e hardware, con dummy `software` e interazione tra `debt_equity_medio` e settore software.

In [ ]:
software_df = software_df.assign(software=1)
hardware_df = hardware_df.assign(software=0)

df_pooled = pd.concat([software_df, hardware_df], ignore_index=True)

variabili_modello = [
    "roe_medio",
    "debt_equity_medio",
    "liquidita_media",
    "log_attivo_medio",
    "eta_azienda",
    "software",
]

df_ols = df_pooled[variabili_modello].dropna().copy()

formula = (
    "roe_medio ~ debt_equity_medio + liquidita_media + log_attivo_medio "
    "+ eta_azienda + software + debt_equity_medio:software"
)

modello_ols = smf.ols(formula=formula, data=df_ols).fit()

print(f"Osservazioni pooled: {len(df_pooled)}")
print(f"Osservazioni usate nella regressione: {len(df_ols)}")
print(modello_ols.summary())

### **TEST DI BREUSCH-PAGAN**

Test di eteroschedasticità sui residui della regressione OLS pooled.

In [ ]:
bp_test = het_breuschpagan(modello_ols.resid, modello_ols.model.exog)

bp_results = pd.Series(
    bp_test,
    index=[
        "LM statistic",
        "LM p-value",
        "F statistic",
        "F p-value",
    ],
)

print("Test di Breusch-Pagan sui residui OLS")
print(bp_results)


### **REGRESSIONE OLS POOLED CON ERRORI STANDARD ROBUSTI HC3**

A seguito dell'evidenza di eteroschedasticità, il modello viene ristimato con errori standard robusti HC3.

In [ ]:
modello_roe_pooled = smf.ols(formula=formula, data=df_ols).fit(cov_type="HC3")

print(modello_roe_pooled.summary())

### **ESPORTAZIONE TABELLA COEFFICIENTI IN LATEX**

In [ ]:
export_dir = Path("regressione_export/tex")
export_dir.mkdir(parents=True, exist_ok=True)

conf_int = modello_roe_pooled.conf_int()
tabella_coefficienti = pd.DataFrame(
    {
        "Coefficiente": modello_roe_pooled.params,
        "Errore standard HC3": modello_roe_pooled.bse,
        "t": modello_roe_pooled.tvalues,
        "p-value": modello_roe_pooled.pvalues,
        "IC 95% inf.": conf_int[0],
        "IC 95% sup.": conf_int[1],
    }
)

latex_path = export_dir / "modello_roe_pooled.tex"
tabella_coefficienti.to_latex(
    latex_path,
    float_format="%.4f",
    caption="Coefficienti del modello OLS pooled sul ROE con errori standard robusti HC3",
    label="tab:modello_roe_pooled",
)

print(f"Tabella esportata in: {latex_path}")
display(tabella_coefficienti)
